In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Documentos\ML_Regression_Model\Regression_ML_EndtoEnd\.venv\Scripts\python.exe
3.0.4
c:\Documentos\ML_Regression_Model\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Documentos\ML_Regression_Model\Regression_ML_EndtoEnd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ==========================================
# 2. Load processed datasets
# ==========================================
train_df = pd.read_csv("C:/Documentos/ML_Regression_Model/Regression_ML_EndtoEnd/data/processed/feature_engineered_train.csv")
eval_df = pd.read_csv("C:/Documentos/ML_Regression_Model/Regression_ML_EndtoEnd/data/processed/feature_engineered_eval.csv")

# ==========================================
# Definición de Target, Features y PURGA DE LEAKAGE
# ==========================================
target = "price"

# Lista de columnas que hacen trampa
leaky_features = ["median_list_price", "Median Home Value", "median_ppsf", "median_list_ppsf"]

# Separamos X e y
X_train_raw = train_df.drop(columns=[target])
y_train = train_df[target]

X_eval_raw = eval_df.drop(columns=[target])
y_eval = eval_df[target]

# Eliminamos la fuga de datos directamente de la raíz
X_train = X_train_raw.drop(columns=leaky_features, errors='ignore')
X_eval = X_eval_raw.drop(columns=leaky_features, errors='ignore')

print("Train shape (Clean):", X_train.shape)
print("Eval shape (Clean):", X_eval.shape)

Train shape (Clean): (576815, 35)
Eval shape (Clean): (148448, 35)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [7]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("/Documentos/ML_Regression_Model/Regression_ML_EndtoEnd//mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/03/09 20:31:29 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-03-09 20:31:29,851] A new study created in memory with name: no-name-d1583217-f63e-46b2-92cb-c0a811bd564c
[I 2026-03-09 20:31:41,694] Trial 0 finished with value: 116893.71906584183 and parameters: {'n_estimators': 487, 'max_depth': 10, 'learning_rate': 0.06719640660254252, 'subsample': 0.6832892649589359, 'colsample_bytree': 0.5122105293252583, 'min_child_weight': 9, 'gamma': 2.9001515331447107, 'reg_alpha': 4.882485597228863e-07, 'reg_lambda': 0.10473884256596418}. Best is trial 0 with value: 116893.71906584183.
[I 2026-03-09 20:31:48,348] Trial 1 finished with value: 119296.13931500331 and parameters: {'n_estimators': 873, 'max_depth': 5, 'learning_rate': 0.29387068689431617, 'subsample': 0.5230659352682621, 'colsample_bytree': 0.8667727125842697, 'min_child_weight': 4, 'gamma': 4.868132811940152, 'reg_alpha': 0.8528897607073435, 'reg_lamb

Best params: {'n_estimators': 996, 'max_depth': 8, 'learning_rate': 0.09031349946926502, 'subsample': 0.8827844124218929, 'colsample_bytree': 0.7537018261297621, 'min_child_weight': 1, 'gamma': 4.032321981851658, 'reg_alpha': 0.0132725268017627, 'reg_lambda': 1.9161386923817324e-08}


Nota Operativa: 15 trials es un número muy bajo para un espacio de búsqueda con 8 hiperparámetros. En un entorno de producción, dejaríamos esto corriendo con n_trials=100 o n_trials=200 durante la noche.

In [8]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 57733.455536531015
RMSE: 112748.22609023559
R²: 0.9017620103268016


c:\Documentos\ML_Regression_Model\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [20:34:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/03/09 20:34:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/09 20:34:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
